# SLM Evaluation Analysis

Three open-weight SLMs evaluated on three production-relevant tasks:
- **Structured Extraction** — field-level F1 on synthetic job postings
- **RAG Q&A** — exact match + F1 on SQuAD (passage provided, not open-domain)
- **Intent Classification** — accuracy on Banking77 (77-class)


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

RESULTS_DIR = Path("../results")
df = pd.read_csv(RESULTS_DIR / "summary.csv")
df.head()

## 1. Accuracy by Model and Task

In [ ]:
# Primary accuracy metric differs by task
metric_map = {
    "extraction": "f1",
    "rag_qa": "f1",
    "classification": "accuracy",
}

rows = []
for _, row in df.iterrows():
    metric = metric_map[row["task"]]
    rows.append({"model": row["model"], "task": row["task"], "score": row[metric], "metric": metric})
plot_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=plot_df, x="task", y="score", hue="model", ax=ax)
ax.set_title("Primary accuracy metric by model and task")
ax.set_ylabel("Score (F1 or Accuracy)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 2. Throughput vs. Accuracy

Each point is one model × task combination. Point size reflects peak memory usage.

In [ ]:
scatter_df = plot_df.merge(df[["model", "task", "tokens_per_sec", "peak_memory_gb"]], on=["model", "task"])

fig, ax = plt.subplots(figsize=(8, 6))
models = scatter_df["model"].unique()
markers = ["o", "s", "^"]  # one per model

for model, marker in zip(models, markers):
    sub = scatter_df[scatter_df["model"] == model]
    ax.scatter(
        sub["tokens_per_sec"],
        sub["score"],
        s=sub["peak_memory_gb"] * 60,
        label=model,
        marker=marker,
        alpha=0.8,
    )

ax.set_xlabel("Tokens / second")
ax.set_ylabel("Score")
ax.set_title("Throughput vs. accuracy (bubble size = peak memory GB)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Extraction: Per-Field F1 Breakdown

Which fields does each model struggle with?

In [ ]:
FIELDS = ["job_title", "company", "location", "salary_range", "years_experience_required"]
MODELS = ["qwen", "phi", "llama"]

per_field_rows = []
for model in MODELS:
    raw_path = RESULTS_DIR / "raw" / f"{model}_extraction.json"
    if not raw_path.exists():
        continue
    with open(raw_path) as f:
        data = json.load(f)
    for field in FIELDS:
        tp = sum(1 for r in data["raw"] if r["per_field"].get(field) == "tp")
        total = sum(1 for r in data["raw"] if r["per_field"].get(field) is not None)
        per_field_rows.append({"model": model, "field": field, "accuracy": tp / total if total > 0 else 0})

pf_df = pd.DataFrame(per_field_rows)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=pf_df, x="field", y="accuracy", hue="model", ax=ax)
ax.set_title("Extraction: per-field accuracy by model")
ax.set_ylim(0, 1)
ax.set_xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. Classification: Confusion Heatmap

Which intents are most frequently confused?

In [ ]:
# Load all classification results and find intent groups with lowest accuracy per model
for model in MODELS:
    raw_path = RESULTS_DIR / "raw" / f"{model}_classification.json"
    if not raw_path.exists():
        continue
    with open(raw_path) as f:
        data = json.load(f)

    by_intent = {}
    for r in data["raw"]:
        intent = r["gold"]
        by_intent.setdefault(intent, {"correct": 0, "total": 0})
        by_intent[intent]["total"] += 1
        by_intent[intent]["correct"] += int(r["correct"])

    intent_acc = {k: v["correct"] / v["total"] for k, v in by_intent.items()}
    worst = sorted(intent_acc.items(), key=lambda x: x[1])[:10]

    print(f"\n{model} — 10 hardest intents:")
    for intent, acc in worst:
        print(f"  {intent:<45} {acc:.0%}")

## 5. Key Findings

> **TODO:** Write 1–2 sentence thesis after reviewing the results above.

Example framing:
- *Phi-3.5-mini matches Llama-3B on structured extraction while running 2x faster, suggesting it is the better default for document parsing pipelines.*
- *All three models struggle with salary normalization (lowest per-field F1), indicating that numeric range extraction benefits from post-processing heuristics.*


## 6. Limitations & Next Steps

- **Prompt sensitivity:** Results will shift with prompt wording; no prompt tuning was done.
- **Extraction dataset size:** 20 synthetic job postings is enough to show the pattern but too small for statistical confidence. A next step would be generating 150+ examples with a larger LM.
- **Few-shot count:** Banking77 classification used 5 random examples. Sweeping N (0, 5, 10, 20) would show how much few-shot helps each model.
- **Quantization:** Running 4-bit quantized versions of the larger models would sharpen the accuracy-vs-memory tradeoff story.
